In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (MultiPep)

This notebook curates the **MultiPep** source by integrating peptide sequences from multiple underlying databases and tool-specific collections. The goal is to build standardized, task-specific datasets (hemolytic, toxic, embryotoxic), apply duplicate consistency checks within each task, and export curated datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** hemolytic, toxic, and embryotoxic
- **Source:** Karasev et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses multiple sub-sources included in MultiPep**, each with its own file format and labeling conventions:
  - **APD3_data**: parses text exports (e.g., hemolytic) and assigns positive labels (`label = 1`).
  - **BIOPEP-UWM**: loads tabular data and extracts subsets for hemolytic, toxic, and embryotoxic activities.
  - **peptide_db**: loads toxic sequences from a plain-text list.
  - **sequences_for_tools**: parses FASTA-style inputs where activity is inferred from record identifiers (e.g., `toxic`, `hemolytic` tokens).
- **Builds task-specific datasets** by concatenating curated subsets:
  - `hemolytic` (positive-only)
  - `toxic` (positive-only)
  - `embryotoxic` (positive-only; derived from BIOPEP-UWM)
- **Checks duplicated sequences** independently per task and for the modified subset:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - conflicting duplicates are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet (matching multiple rows for the MultiPep umbrella source) and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`
  - `processed_toxic_dataset.csv`
  - `processed_embryotoxic_dataset.csv`
  - `detected_error_sequences.csv`
  - `detected_error_modified_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "MultiPep"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

##### Parsing APD3_data [multipep]

- Reading raw data

In [3]:
sub_source = f"{name_source}/APD3_data"

In [4]:
df_hemolytic_apd3 = read_apd3_txt(PATH_INPUT, sub_source, "hemolytic.txt")
df_hemolytic_apd3["label"] = 1
df_hemolytic_apd3 = df_hemolytic_apd3[["sequence", "label"]]

##### Parsing BIOPEP-UWM [multipep]

- Reading raw data

In [5]:
sub_source = f"{name_source}/BIOPEP-UWM"
df_biopep = pd.read_csv(f"{PATH_INPUT}/{sub_source}/biopep_data.txt", sep="\t")

In [6]:
df_biopep = (
    df_biopep
    .rename(columns={"Sequence": "sequence", "Activity ": "label"})
    .assign(
        label=lambda d: d["label"].str.strip().str.lower(),
        sequence=lambda d: d["sequence"].str.replace("~", "", regex=False)
    )
)

- Separate dataset by toxic effects

In [7]:
df_hemolytic_biopep = (
    df_biopep
    .assign(
        label=lambda d: d["label"]
        .str.contains("haemolytic", case=False, na=False)
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)
df_hemolytic_biopep.shape

(61, 2)

In [8]:
df_toxic_biopep = (
    df_biopep
    .loc[
        df_biopep["label"].str.lower().str.strip() == "toxic",
        ["sequence", "label"]
    ]
    .assign(label=1)
    .reset_index(drop=True)
)
df_toxic_biopep.shape

(9, 2)

In [9]:
df_embryotoxic_biopep = (
    df_biopep
    .assign(
        label=lambda d: d["label"]
        .str.contains("embryotoxic", case=False, na=False)
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)
df_embryotoxic_biopep.shape

(3, 2)

##### Parsing DBAASP [multipep]

- Reading raw data

In [10]:
sub_source = f"{name_source}/DBAASP"

##### Parsing peptide_db [multipep]

- Reading raw data

In [11]:
sub_source = f"{name_source}/peptide_db"
df_toxic_peptidedb = pd.read_csv(f"{PATH_INPUT}/{sub_source}/tox_ven.txt", names=["sequence"])
df_toxic_peptidedb["label"] = 1

##### Parsing SATPdb [multipep]

- Reading raw data

In [12]:
sub_source = f"{name_source}/SATPdb"

In [13]:
df_toxic_satpdb = (
    read_fasta_with_strange_character(f"{PATH_INPUT}/{sub_source}/toxic.fa")
    .assign(label=1)
    [["sequence", "label"]]
)

##### Parsing sequences_for_tools [multipep]

In [14]:
sub_source = f"{name_source}/sequences_for_tools"

In [15]:
df_toxic_for_tools = read_fasta_doc(f"{PATH_INPUT}/{sub_source}/to_toxinpred.txt")

In [16]:
df_hplpred_for_tools = read_fasta_doc(f"{PATH_INPUT}/{sub_source}/to_hplpred.txt")

In [17]:
df_toxic_for_tools = (
    df_toxic_for_tools
    .assign(
        label=lambda d: d["id"]
        .str.contains("toxic", case=False, na=False)
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)
df_toxic_for_tools.shape

(459, 2)

In [18]:
df_hemolytic_hplpred = (
    df_hplpred_for_tools
    .assign(
        label=lambda d: d["id"]
        .str.contains("hemolytic", case=False, na=False)
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)
df_hemolytic_hplpred.shape

(130, 2)

- Concatenating dataset

In [19]:
df_hemolytic = pd.concat([
    df_hemolytic_hplpred,
    df_hemolytic_apd3,
    df_hemolytic_biopep],
    ignore_index=True
)
df_hemolytic.shape

(504, 2)

In [20]:
df_toxic = pd.concat([
    df_toxic_biopep,
    df_toxic_for_tools,
    df_toxic_peptidedb,
    df_toxic_satpdb],
    ignore_index=True
)
df_toxic.shape

(7085, 2)

- Checking duplicates

In [21]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [22]:
df_remove_duplicated_toxic, df_errors_toxic, df_unique_toxic = processing_duplicated(df_toxic, group_seq="sequence", sort_key="label")

In [23]:
df_remove_duplicated_embryo, df_errors_embryo, df_unique_embryo = processing_duplicated(df_embryotoxic_biopep, group_seq="sequence", sort_key="label")

In [24]:
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])
df_full_toxic = pd.concat([df_unique_toxic, df_remove_duplicated_toxic])
df_full_embryo = pd.concat([df_remove_duplicated_embryo, df_unique_embryo])
df_full = pd.concat([df_full_hemolytic, df_full_toxic, df_full_embryo])
df_errors = pd.concat([df_errors_hemolytic, df_errors_toxic, df_errors_embryo])

- Working with metada

In [25]:
df_metada = read_metadata_multiple("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada) 

In [26]:
raw_total_sequences = (
    len(df_hemolytic_apd3)
    + len(df_biopep)
    + len(df_toxic_peptidedb)
    + len(df_toxic_satpdb)
    + len(df_toxic_for_tools)
    + len(df_hplpred_for_tools)
)

In [27]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Dynamic',
 'license': 'MIT',
 'year of publication': 2021,
 'last update date': datetime.datetime(2023, 3, 30, 0, 0),
 'download date': '2025-04-01 00:00:00;2025-08-01 00:00:00',
 'file format': 'txt;tsv;csv;fasta',
 'peptide property': 'hemolytic, toxic;insecticidal, toxic;ACE inhibitor, antioxidant, antibacterial, antimicrobial, dipeptidyl peptidase IV inhibitor, celiac toxic, opioid, neuropeptides, immunomodulating, dipeptidyl peptidase III inhibitor, antithrombotic, antifungal, antiamnestic, hemolytic, toxic, anticancer, CaMKII Inhibitor, alpha-glucosidase inhibitor, HMG-CoA reductase inhibitor, binding peptides, heparin binding, antiviral, renin inhibitor, immunostimulating, immunogenic peptides, antihypertensive, alpha-amylase inhibitor, opioid agonist, membrane -active , antiinflammatory, CaMPDE inhibitor, hypolipidemic, chemotactic, opioid antagonist, antidiabetic, vasoconstrictor, bacterial permease ligand, activating ubiquitin-me

- Exporting data

In [28]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [29]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_toxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_full_embryo.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_embryotoxic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)